# SOHO 영업지속 평점모형 개발

12개월 영업지속 라벨과 독립변수로 **WOE → 로지스틱 → 확률 보정 → 점수·등급**을 산출합니다. `BAD = 1 - Y_12M`이며, BAD는 폐업을 뜻합니다. 대출 부도나 개별 점포 매출 회복을 측정하지 않습니다.

이 노트북은 `src/models`의 학습 코드를 실행하고 CSV·JSON 보고서를 확인하는 순서로 구성했습니다. 기본값은 **기존 결과 조회**입니다. 다시 학습하려면 아래 `RUN_TRAINING`을 `True`로 바꾸세요.

- 원본 252,424행 → 완전 중복 제거 후 32,965개 사업자.
- 학습 2019–2020 / 검증 2022 / OOT 2024. 2021·2023은 관측기간 간격을 확보합니다.
- 폐업 관측 종료일 `2025-12-31`은 확인된 사실이 아닌 승인된 가정입니다.
- 기준일을 확인하지 못한 추가 인구·지하철 변수는 기본 학습에서 제외합니다.

자세한 설정·산출물·검토 항목: [개발 안내](../docs/scorecard.md). 원본 노트북은 `notebooks/archive`에 보존했습니다.


In [ ]:
from pathlib import Path
import json
import os
import sys
import pandas as pd
from IPython.display import display, Markdown, JSON

candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "iM_Blockchain_AI"]
PROJECT_ROOT = next(
    (path for path in candidates
     if (path / "src" / "data").is_dir() and (path / "notebooks").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("프로젝트 루트 또는 notebooks 폴더에서 실행하세요.")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 100)
print("프로젝트:", PROJECT_ROOT)


## 1. 실행 설정

모델 전용 `.venv-model` 환경을 커널로 사용하세요. 환경 준비 방법은 개발 안내를 참고하세요. 기간·통계 임계값은 `ScorecardConfig`에서 관리합니다. 현재 입력에는 `Y_24M`이 없으므로 24개월로 바꾸려면 라벨을 먼저 준비해야 합니다.


In [ ]:
from dataclasses import asdict
from src.models.config import ScorecardConfig
from src.models.train_scorecard import run_pipeline

RUN_TRAINING = False
config = ScorecardConfig()
display(JSON(json.loads(json.dumps(asdict(config), default=str)), expanded=False))


In [ ]:
latest_path = PROJECT_ROOT / "data" / "processed" / "scorecard" / "latest_run.json"
if RUN_TRAINING:
    result = run_pipeline(config)
elif latest_path.exists():
    result = json.loads(latest_path.read_text(encoding="utf-8-sig"))
else:
    result = {}
    print("저장된 실행 결과가 없습니다. RUN_TRAINING = True로 실행하세요.")

def resolve_output(value):
    if not value:
        return None
    path = Path(value)
    return path if path.is_absolute() else PROJECT_ROOT / path

REPORT_DIR = resolve_output(result.get("report_dir"))
MODEL_DIR = resolve_output(result.get("model_dir"))
print("보고서:", REPORT_DIR)
print("모형:", MODEL_DIR)
if result.get("summary"):
    display(JSON(result["summary"], expanded=False))


In [ ]:
def show_report(name, limit=30):
    """저장된 전체 CSV에서 일부 행을 조회합니다. 원본 파일에는 전체 결과가 있습니다."""
    if REPORT_DIR is None:
        return None
    path = REPORT_DIR / f"{name}.csv"
    if not path.exists():
        print(f"{name}: 보고서 파일이 없습니다.")
        return None
    try:
        table = pd.read_csv(path, encoding="utf-8-sig")
    except pd.errors.EmptyDataError:
        print(f"{name}: 기록할 행이 없습니다.")
        return pd.DataFrame()
    display(Markdown(f"**{name}** — {len(table):,}행 · 전체 결과: `{path}`"))
    display(table.head(limit))
    return table

def show_reports(*names):
    for name in names:
        show_report(name)


## 2. 데이터 검증과 시점 분할

중복 제거 전후 분포를 확인합니다. 기준일과 라벨 종료일로 분할하고, 학습 시점 이후 관측된 정보가 학습에 섞이지 않도록 합니다. 추가 변수의 원천·집계일·공개일을 확인하기 전에는 기본 후보에 포함하지 않습니다.


In [ ]:
show_reports("data_audit", "feature_inventory", "split_summary")


## 3. 변수 검증과 WOE 구간화

구간 경계와 WOE는 학습 데이터만으로 추정합니다. 기본 선별 기준은 IV 0.02, 결측률 0.95, 최빈값 비율 0.95입니다. 높은 IV나 단조성만으로 변수의 업무적 타당성이 보장되지는 않습니다.


In [ ]:
show_reports("feature_screening", "woe_summary", "woe_fine", "woe_coarse")


## 4. 상관성 제거와 최종 변수 선택

상관계수 절댓값 0.7 기준과 L1 로지스틱으로 후보를 줄입니다. 정규화 강도는 검증 집합에서 선택합니다. `selection_log`에서 변수별 포함·제외 사유를 확인하세요.


In [ ]:
show_reports("correlation_pairs", "selection_log", "c_tuning", "coefficients")


## 5. 확률 보정과 모형 검증

검증기간을 시간순으로 나누어 확률 보정의 개선 여부를 비교한 뒤 채택 여부를 결정합니다. 검증 집합은 선택·보정에도 사용되므로 최종 성능은 OOT에서 판단합니다.


In [ ]:
show_reports("calibration_comparison", "calibration_curve", "metrics", "metrics_by_year")


## 6. 배점과 등급

기준점수 600, good:bad odds 20:1, PDO 50을 사용합니다. 높은 점수는 낮은 폐업 위험을 뜻합니다. 등급은 검증 데이터에서 최대 5개로 정하고 OOT에 고정 적용합니다. 등급별 표본 수·폐업률·순서·분포 안정성을 함께 확인하세요.


In [ ]:
show_reports("scorecard", "grade_definition", "grade_performance", "score_variance")


In [ ]:
show_reports("psi_summary", "psi_detail")


## 7. 사업자별 산출 결과와 검토 사항

사업자별 확률·점수·등급 및 변수별 점수 기여도는 아래 파일에서 확인합니다. 화면은 일부 행만 보여주며 CSV·JSON에는 전체 결과를 저장합니다.

사람이 확인할 사항은 **폐업 관측 완전성, 추가 변수의 시점과 병합 근거, 변수의 업무적 의미, 등급별 지원 기준, 실제 매출로 교체한 뒤 운영 적용 여부**입니다. 현재 합성 매출을 이용한 결과는 PoC입니다.


In [ ]:
show_reports("scored_businesses", "score_contributions", "human_review", "run_summary")


## 저장된 그래프
전체 이미지 목록은 figure_manifest.csv/json에서 확인합니다. 아래는 주요 결과입니다.

In [ ]:
from IPython.display import Image
show_report("figure_manifest")
if REPORT_DIR is not None:
    for filename in ["10_discrimination_oot.png", "11_calibration.png", "13_grades.png"]:
        path = REPORT_DIR / "figures" / filename
        if path.exists():
            display(Image(filename=str(path)))
